In [ ]:
import requests
from pathlib import Path

# Kept under the upstream names so configs/base.yaml points at the files this
# notebook actually writes. The test split is product-disjoint from both train
# and validation, so validation is used whole rather than being carved in half.
urls = {
    "train-qar.jsonl": "https://amazon-qa.s3-us-west-2.amazonaws.com/train-qar.jsonl",
    "val-qar.jsonl": "https://amazon-qa.s3-us-west-2.amazonaws.com/val-qar.jsonl",
    "test-qar_all.jsonl": "https://amazon-qa.s3-us-west-2.amazonaws.com/test-qar_all.jsonl",
}

for name, url in urls.items():
    if Path(name).exists():
        print(f"Present, skipping: {name}")
        continue

    # Streamed: these are 0.75-2.7 GB, and response.content holds the whole file
    # in memory before a single byte reaches disk.
    with requests.get(url, stream=True, timeout=300) as response:
        response.raise_for_status()
        with open(name, "wb") as file:
            for chunk in response.iter_content(chunk_size=1 << 20):
                file.write(chunk)

    print(f"Downloaded: {name} ({Path(name).stat().st_size / 1e9:.2f} GB)")